<a href="https://colab.research.google.com/github/kiato1970-blip/AI_EKPA/blob/main/OnlineLearning_IPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#εισαγωγή βιβλιοθηκών - πακέτων
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
import requests
from io import BytesIO
import gzip
import numpy as np


In [10]:
import requests
from io import BytesIO
import gzip
import pandas as pd

# ΤΟ ΣΩΣΤΟ URL ΓΙΑ ΤΟ DATASET
url = "https://uci.edu"

# Προσθέτουμε User-Agent για να μην τρώμε πόρτα (403 Error)
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    # Αποσυμπίεση και διάβασμα
    with gzip.open(BytesIO(response.content), 'rb') as f:
        df = pd.read_csv(f, header=None)
        print("Επιτυχής λήψη!")
        print(df.head()) # Εμφάνιση των πρώτων γραμμών

except Exception as e:
    print(f"Προέκυψε σφάλμα: {e}")

Προέκυψε σφάλμα: Not a gzipped file (b'<!')


In [16]:
import pandas as pd
from sklearn.datasets import fetch_kddcup99

print("Γίνεται λήψη των δεδομένων (KDD Cup 99 - 10%)...")

try:
    # Φόρτωση του dataset
    dataset = fetch_kddcup99(subset=None, percent10=True, as_frame=True)
    df = dataset.frame

    # Ασφαλής μετατροπή bytes σε strings μόνο όπου χρειάζεται
    for col in df.columns:
        # Ελέγχουμε αν τα περιεχόμενα είναι bytes (π.χ. b'tcp')
        if df[col].dtype == object:
            # Δοκιμάζουμε να δούμε αν το πρώτο στοιχείο είναι bytes
            first_val = df[col].iloc[0]
            if isinstance(first_val, bytes):
                df[col] = df[col].str.decode('utf-8')

    print("Επιτυχής φόρτωση!")
    print(f"Διαστάσεις: {df.shape}")
    print("\nΠρώτες γραμμές:")
    print(df.head())

except Exception as e:
    print(f"Προέκυψε σφάλμα: {e}")

Γίνεται λήψη των δεδομένων (KDD Cup 99 - 10%)...
Επιτυχής φόρτωση!
Διαστάσεις: (494021, 42)

Πρώτες γραμμές:
  duration protocol_type service flag src_bytes dst_bytes land wrong_fragment  \
0        0           tcp    http   SF       181      5450    0              0   
1        0           tcp    http   SF       239       486    0              0   
2        0           tcp    http   SF       235      1337    0              0   
3        0           tcp    http   SF       219      1337    0              0   
4        0           tcp    http   SF       217      2032    0              0   

  urgent hot  ... dst_host_srv_count dst_host_same_srv_rate  \
0      0   0  ...                  9                    1.0   
1      0   0  ...                 19                    1.0   
2      0   0  ...                 29                    1.0   
3      0   0  ...                 39                    1.0   
4      0   0  ...                 49                    1.0   

  dst_host_diff_srv_rate 

In [18]:
# Έλεγχος για το σωστό όνομα της στήλης-στόχου
target_col = 'labels' if 'labels' in df.columns else 'target'

# Δημιουργία X και y
X = df.drop(target_col, axis=1)

# Μετατροπή σε 0 (Normal) και 1 (Attack)
# Χρησιμοποιούμε το .astype(str) για να σιγουρευτούμε ότι η σύγκριση με το "normal." θα πιάσει
y = df[target_col].astype(str).apply(lambda x: 0 if 'normal' in x else 1)

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

X shape: (494021, 41)
y distribution:
 labels
1    396743
0     97278
Name: count, dtype: int64


In [19]:
#έλεγχος κλάσεων
print("Κατανομή κλάσεων y:")
print(y.value_counts())

Κατανομή κλάσεων y:
labels
1    396743
0     97278
Name: count, dtype: int64


In [20]:
#εντοπισμός κατηγορικών μεταβλητών
categorical_features = ['protocol_type', 'service', 'flag']

In [21]:
#διαχωρισμός κατηγορικών και αριθμητικών μεταβλητών
numeric_features = X.columns.difference(categorical_features)

In [22]:
#δημιουργία διοχέτευσης (αγωγού) προεπεξεργασίας με κωδικοποίηση μίας δέσμης (one-hot encoding) για κατηγορικές μεταβλητές
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [23]:
#τμηματοποίηση δεδομένων σε σύνολα εκαπίδευσης και δοκιμών
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [24]:
#δημιουργία διοχέτευσης (αγωγού) SMOTE για τις αριθμητικές μεταβλητές μόνο
pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTENC(random_state=42, categorical_features=[X.columns.get_loc(col) for col in categorical_features])),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [25]:
#καθορισμός κατωφλίου (threshold) για την διακοπή διακτυακής κίνησης
blocking_threshold = 0.9

In [26]:
#δημιουργία βρόχου συνεχούς - αυξητικής μάθησης
batch_size = 10000
for epoch in range(1, 3):  #δυνατότητα αλλαγής των εποχών
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train.iloc[i:i + batch_size]
        y_batch = y_train.iloc[i:i + batch_size]

        #σταδιακή ενημέρωση του μοντέλου με κάθε ροή (batch) δεδομένων
        pipeline.fit(X_batch, y_batch)

        #περιοδική ενημέρωση του μοντέλου στο σύνολο δοκιμών
        if i % batch_size == 0 and i > 0:
            y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

            #αποκλεισμός δικτυακής κυκλοφορίας εάν η προβλεπόμενη πιθανότητα υπερβαίνει το καθορισμένο όριο
            blocked_indices = np.where(y_pred_proba > blocking_threshold)[0]
            if len(blocked_indices) > 0:
                print(f"Blocking {len(blocked_indices)} malicious traffic instances.")

            accuracy = accuracy_score(y_test, y_pred_proba > blocking_threshold)
            print(f"Epoch {epoch}, Iteration {i}, Test Accuracy: {accuracy}")

Blocking 118380 malicious traffic instances.
Epoch 1, Iteration 10000, Test Accuracy: 0.9957154520366784
Blocking 118206 malicious traffic instances.
Epoch 1, Iteration 20000, Test Accuracy: 0.9945414184215321
Blocking 118321 malicious traffic instances.
Epoch 1, Iteration 30000, Test Accuracy: 0.995317360178669
Blocking 118405 malicious traffic instances.
Epoch 1, Iteration 40000, Test Accuracy: 0.9958841350273604
Blocking 118424 malicious traffic instances.
Epoch 1, Iteration 50000, Test Accuracy: 0.9960123341002787
Blocking 118137 malicious traffic instances.
Epoch 1, Iteration 60000, Test Accuracy: 0.9940758533672499
Blocking 118123 malicious traffic instances.
Epoch 1, Iteration 70000, Test Accuracy: 0.993981390892468
Blocking 118434 malicious traffic instances.
Epoch 1, Iteration 80000, Test Accuracy: 0.9960798072965514
Blocking 118330 malicious traffic instances.
Epoch 1, Iteration 90000, Test Accuracy: 0.9953780860553145
Blocking 118272 malicious traffic instances.
Epoch 1, Ite

In [27]:
#τελική αξιολόγηση του μοντέλου στο σύνολο δοκιμών
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
blocked_indices = np.where(y_pred_proba > blocking_threshold)[0]
if len(blocked_indices) > 0:
    print(f"Blocking {len(blocked_indices)} malicious traffic instances.")

accuracy = accuracy_score(y_test, y_pred_proba > blocking_threshold)
classification_rep = classification_report(y_test, y_pred_proba > blocking_threshold)

#εκτύπωση τελικών αποτελεσμάτων
print(f"Final Test Accuracy: {accuracy}")
print("Classification Report:")
print(classification_rep)

Blocking 118164 malicious traffic instances.
Final Test Accuracy: 0.9942580309971863
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99     29192
           1       1.00      0.99      1.00    119015

    accuracy                           0.99    148207
   macro avg       0.99      1.00      0.99    148207
weighted avg       0.99      0.99      0.99    148207

